# 📊 Strongbox Parser - Google Colab Edition

**Convert Strongbox Excel files to Audit Sight format - No installation required!**

## 🚀 Instructions:
1. **Click Runtime → Run all** (or press Ctrl+F9)
2. **Wait for setup** to complete (about 30 seconds)
3. **Upload your file** when prompted
4. **Download** the processed result

**Features:**
- ✅ Processes multiple TXN-FY sheets with real transaction data
- ✅ Auto-detects date ranges from TB data
- ✅ Creates Comparative Trial Balance from your accounts
- ✅ Generates Journal Entries & Lines from your transactions
- ✅ Professional Excel formatting
- ✅ Data cleaning and Unicode handling
- ✅ Complete with all required tabs (Instructions, Banking, etc.)
- ✅ **NEW: Uses TB-DATA sheet for accurate balance information**

**Team-friendly: Share this link with anyone who needs to process Strongbox files!**

In [ ]:
#@title 🔧 Setup (Run this first)
print('🔧 Installing required packages...')
!pip install -q openpyxl python-dateutil xlsxwriter
print('✅ Setup complete! Ready to process files.')

In [ ]:
#@title 📚 Import Libraries
import pandas as pd
import os
from datetime import datetime
from dateutil.relativedelta import relativedelta
import calendar
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment
from copy import copy
import math
import sys
from google.colab import files
import io
import re
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries loaded successfully!')

In [ ]:
#@title 🏗️ Load Strongbox Parser
class StrongboxParserColab:
    def __init__(self):
        self.source_file = None
        self.start_date = None
        self.end_date = None
        self.start_date = None
        self.source_data = {}
        self.template_data = {}
        self.output_filename = None
        self.date_columns = {}

    def print_and_log(self, message):
        print(message)

    def update_status(self, message, progress=None):
        if progress:
            print(f'[{progress:2d}%] {message}')
        else:
            print(message)

    def determine_date_range(self):
        self.update_status('Determining date range from TB sheet...', 10)
        
        date_row = pd.read_excel(self.source_file, sheet_name='TB', header=None, nrows=1, skiprows=3)
        date_row = date_row.iloc[0]
        
        date_columns = {}
        for col_idx, value in enumerate(date_row):
            try:
                if pd.notna(value):
                    if isinstance(value, str):
                        for fmt in ['%m/%d/%Y', '%Y-%m-%d', '%m-%d-%Y', '%d-%b-%Y', '%d/%b/%Y']:
                            try:
                                date = datetime.strptime(value, fmt)
                                date_columns[date] = col_idx
                                break
                            except ValueError:
                                continue
                    else:
                        date = pd.to_datetime(value)
                        date_columns[date] = col_idx
            except Exception:
                continue
        
        if not date_columns:
            raise Exception('No valid dates found in TB sheet row 4')
        
        tb_dates = sorted(date_columns.keys())
        if len(tb_dates) < 2:
            raise Exception('Need at least two dates in TB sheet')
        
        self.begin_balance_date = tb_dates[0]
        self.start_date = tb_dates[0] + relativedelta(days=1)
        self.end_date = tb_dates[-1]
        self.date_columns = date_columns
        
        self.print_and_log(f'📅 Date range: {self.start_date.strftime("%Y-%m-%d")} to {self.end_date.strftime("%Y-%m-%d")}')
        self.print_and_log(f'📅 Date columns mapping: {[(d.strftime("%Y-%m-%d"), idx) for d, idx in sorted(date_columns.items())]}')
        return date_columns

    def load_source_data(self):
        self.update_status('Loading transaction data...', 20)
        
        excel_file = pd.ExcelFile(self.source_file)
        txn_sheets = [s for s in excel_file.sheet_names if s.startswith('TXN-FY')]
        
        self.print_and_log(f'📊 Found {len(txn_sheets)} transaction sheets: {txn_sheets}')
        
        for sheet_name in txn_sheets:
            try:
                df = pd.read_excel(self.source_file, sheet_name=sheet_name)
                self.print_and_log(f'📋 {sheet_name}: {len(df)} rows loaded')
                
                # Convert date columns using the correct column names
                for col in ['Transaction Date', 'Fiscal Month']:
                    if col in df.columns:
                        df[col] = pd.to_datetime(df[col], errors='coerce')
                
                # Convert numeric columns using the correct column names
                for col in ['Debit', 'Credit']:
                    if col in df.columns:
                        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
                
                # Filter by Fiscal Month date range
                if self.start_date is not None and self.end_date is not None:
                    if 'Fiscal Month' in df.columns:
                        mask = (df['Fiscal Month'] >= self.start_date) & (df['Fiscal Month'] <= self.end_date)
                        df_filtered = df[mask]
                        self.print_and_log(f'📅 {sheet_name}: {len(df_filtered)} transactions in date range')
                        df = df_filtered
                
                if not df.empty:
                    self.source_data[sheet_name] = df
                    self.print_and_log(f'✅ {sheet_name}: {len(df)} transactions loaded')
                
            except Exception as e:
                self.print_and_log(f'⚠️ Error loading {sheet_name}: {str(e)}')

        # Load TB-DATA sheet for balance information
        self.print_and_log("\nLoading TB-DATA sheet...")
        self.update_status("Loading TB-DATA sheet...", 25)
        try:
            tb_data_df = pd.read_excel(
                self.source_file,
                sheet_name='TB-DATA',
                engine='openpyxl',
                na_filter=False,
                keep_default_na=False
            )
            self.source_data['TB-DATA'] = tb_data_df
            self.print_and_log(f"✅ Successfully loaded TB-DATA sheet with {len(tb_data_df)} rows")
            
        except Exception as e:
            self.print_and_log(f"⚠️ Warning: Could not load TB-DATA sheet: {str(e)}")
            self.print_and_log("Will use default balance values (0) if TB-DATA is not available")
            self.source_data['TB-DATA'] = None

    def _extract_balances_from_tb_data(self, account_id, begin_date, end_date):
        """Extract beginning and ending balances from TB-DATA sheet for a specific account"""
        if 'TB-DATA' not in self.source_data or self.source_data['TB-DATA'] is None:
            return 0.0, 0.0
        
        tb_data = self.source_data['TB-DATA']
        
        # Find the account in TB-DATA
        account_data = None
        
        # Check columns B through D (indices 1-3) for Account ID
        for col_idx in [1, 2, 3]:
            try:
                if col_idx < len(tb_data.columns):
                    matching_rows = tb_data[tb_data.iloc[:, col_idx].astype(str) == str(account_id)]
                    if not matching_rows.empty:
                        account_data = matching_rows
                        break
            except Exception:
                continue
        
        if account_data is None or account_data.empty:
            return 0.0, 0.0
        
        begin_balance = 0.0
        end_balance = 0.0
        
        # Find balances matching our target dates
        for _, row in account_data.iterrows():
            try:
                # Column A (index 0) should contain fiscal month
                fiscal_month_val = row.iloc[0]
                if pd.isna(fiscal_month_val):
                    continue
                
                # Try to parse the fiscal month date
                fiscal_month = pd.to_datetime(fiscal_month_val)
                
                # Column G (index 6) is Ending Account Balance
                ending_balance_val = row.iloc[6] if len(row) > 6 else 0.0
                ending_balance = 0.0
                if pd.notna(ending_balance_val):
                    try:
                        ending_balance = float(ending_balance_val)
                    except (ValueError, TypeError):
                        ending_balance = 0.0
                
                # Check if fiscal month matches our target dates
                # Both beginning and ending balance use Ending Account Balance
                if fiscal_month.date() == begin_date.date():
                    begin_balance = ending_balance
                    
                if fiscal_month.date() == end_date.date():
                    end_balance = ending_balance
                    
            except Exception:
                continue
        
        return begin_balance, end_balance

    def load_trial_balance_data(self):
        # This method keeps the existing TB sheet loading logic
        self.update_status('Loading trial balance data...', 40)
        
        # Get the date_columns that were determined in determine_date_range
        date_columns = self.date_columns
        
        # Find the closest TB dates to our calculated range
        available_tb_dates = sorted(date_columns.keys())
        
        # Find closest beginning date
        closest_begin_date = None
        for tb_date in available_tb_dates:
            if tb_date <= self.begin_balance_date:
                closest_begin_date = tb_date
            else:
                break
        
        if closest_begin_date is None:
            closest_begin_date = available_tb_dates[0]
        
        # Find closest ending date
        closest_end_date = None
        for tb_date in reversed(available_tb_dates):
            if tb_date >= self.end_date:
                closest_end_date = tb_date
            else:
                break
        
        if closest_end_date is None:
            closest_end_date = available_tb_dates[-1]
        
        return self._extract_trial_balance_data(date_columns, closest_begin_date, closest_end_date)

    def _extract_trial_balance_data(self, date_columns, closest_begin_date, closest_end_date):
        """Extract data from the TB sheet using openpyxl"""
        try:
            wb = openpyxl.load_workbook(self.source_file, data_only=True, read_only=True)
            tb_sheet = wb['TB']
            
            # Find Financial Statement Classification Path column
            fin_statement_col = self._find_financial_classification_column(tb_sheet)
            
            data = []
            
            # Process rows starting from row 5
            for row_idx in range(5, tb_sheet.max_row + 1):
                try:
                    account_id = tb_sheet.cell(row=row_idx, column=4).value
                    if account_id is not None:
                        account_id = str(account_id).strip()
                        if account_id.lower() in ['account id', 'account']:
                            continue
                            
                        account_name = tb_sheet.cell(row=row_idx, column=6).value
                        fin_statement_class = tb_sheet.cell(row=row_idx, column=fin_statement_col).value
                        
                        if fin_statement_class is None:
                            fin_statement_class = ''
                        else:
                            fin_statement_class = str(fin_statement_class).strip()
                        
                        data.append({
                            'Account Id': account_id,
                            'Account Name': str(account_name).strip() if account_name is not None else '',
                            'Beginning Balance': 0.0,  # Will be updated from TB-DATA
                            'Ending Balance': 0.0,     # Will be updated from TB-DATA
                            'Financial Statement Classification': fin_statement_class
                        })
                except Exception:
                    continue
            
            tb_data = pd.DataFrame(data)
            return tb_data
            
        except Exception as e:
            self.print_and_log(f'Error in _extract_trial_balance_data: {str(e)}')
            raise
        finally:
            try:
                if 'wb' in locals():
                    wb.close()
            except:
                pass

    def _find_financial_classification_column(self, tb_sheet):
        """Find the Financial Statement Classification column in the TB sheet"""
        fin_statement_col = None
        
        for row_idx in range(1, 6):
            for col_idx in range(1, 15):
                try:
                    cell_value = tb_sheet.cell(row=row_idx, column=col_idx).value
                    if cell_value:
                        cell_text = str(cell_value).lower()
                        if 'financial statement classification' in cell_text:
                            fin_statement_col = col_idx
                            return fin_statement_col
                except Exception:
                    continue
        
        if not fin_statement_col:
            fin_statement_col = 3  # Default to column C
        
        return fin_statement_col

    def determine_account_type(self, fs_classification):
        """Determine Account Type based on Financial Statement Classification Path"""
        if pd.isna(fs_classification) or fs_classification == '':
            return ''
            
        fs_classification = str(fs_classification).strip()
        
        if fs_classification.startswith('Total Assets'):
            return 'Assets'
        elif fs_classification.startswith('Total Liabilities and Equity → Total Liabilities'):
            return 'Liabilities'
        elif fs_classification.startswith('Total Liabilities and Equity → Total Equity'):
            return 'Equity'
        elif fs_classification.startswith('Net Income → Operating Profit → Gross Profit → Total Net Sales'):
            return 'Income'
        elif fs_classification.startswith('Net Income → Operating Profit → Gross Profit → Total COGS/COS'):
            return 'Expense'
        elif fs_classification.startswith('Net Income → Operating Profit → Total Operating Expenses'):
            return 'Expense'
        else:
            return ''

    def create_trial_balance(self):
        """Create Comparative Trial Balances tab with TB-DATA balances"""
        self.update_status('Creating trial balance...', 70)
        tb_data = self.source_data['TB']
        
        # Filter out header rows
        tb_data = tb_data[~((tb_data['Account Id'].str.lower() == 'account id') | 
                           (tb_data['Account Id'].str.lower() == 'account'))]
        
        # Create DataFrame with exact column names
        trial_balance = pd.DataFrame({
            'Account ID': tb_data['Account Id'],
            'Account Name': tb_data['Account Name'],
            'Beginning Balance \n(Prior Period Balance)': 0.0,
            'Ending Balance': 0.0,
            'Account Type \n(see Mapping Categories tab)': '',
            'Account Mapping \n(see Mapping Categories tab)': '',
            'Account Description': tb_data['Financial Statement Classification']
        })
        
        # Update balances from TB-DATA sheet
        self.print_and_log("\nUpdating balances from TB-DATA sheet...")
        updated_accounts = 0
        
        for idx, row in trial_balance.iterrows():
            account_id = row['Account ID']
            begin_balance, end_balance = self._extract_balances_from_tb_data(
                account_id, self.begin_balance_date, self.end_date
            )
            
            trial_balance.at[idx, 'Beginning Balance \n(Prior Period Balance)'] = begin_balance
            trial_balance.at[idx, 'Ending Balance'] = end_balance
            
            if begin_balance != 0.0 or end_balance != 0.0:
                updated_accounts += 1
        
        self.print_and_log(f"Updated balances for {updated_accounts} accounts from TB-DATA")
        
        # Apply account type classification
        trial_balance['Account Type \n(see Mapping Categories tab)'] = trial_balance['Account Description'].apply(self.determine_account_type)
        
        # Remove rows with empty Account IDs
        trial_balance = trial_balance[trial_balance['Account ID'].notna() & (trial_balance['Account ID'] != '')]
        trial_balance = trial_balance.reset_index(drop=True)
        
        self.print_and_log(f'✅ Trial balance created: {len(trial_balance)} accounts')
        return trial_balance

    def create_journal_entries(self):
        """Create Journal Entries & Lines tab"""
        self.update_status('Creating journal entries...', 60)
        
        # Get transaction sheets
        transaction_sheets = {k: v for k, v in self.source_data.items() if k.startswith('TXN-FY')}
        
        processed_sheets = []
        for sheet_name, df in transaction_sheets.items():
            try:
                df_copy = df.copy()
                
                # Convert columns to appropriate types
                df_copy['Transaction Id'] = df_copy['Transaction Id'].astype(str)
                df_copy['Memo'] = df_copy['Memo'].fillna('')
                df_copy['Doc/Ref No'] = df_copy['Doc/Ref No'].fillna('')
                df_copy['Account Id'] = df_copy['Account Id'].astype(str)
                
                # Handle optional columns
                for col in ['Transaction Type', 'Relationship Name']:
                    if col in df_copy.columns:
                        df_copy[col] = df_copy[col].fillna('')
                    else:
                        df_copy[col] = ''
                
                # Convert numeric columns
                df_copy['Debit'] = pd.to_numeric(df_copy['Debit'], errors='coerce').fillna(0)
                df_copy['Credit'] = pd.to_numeric(df_copy['Credit'], errors='coerce').fillna(0)
                
                # Create the required columns
                processed_df = pd.DataFrame({
                    'Journal ID': df_copy['Transaction Id'],
                    'Type': df_copy['Transaction Type'],
                    'Journal Entry Description': df_copy['Doc/Ref No'],
                    'Posted Date': df_copy['Transaction Date'],
                    'Account ID': df_copy['Account Id'],
                    'Journal Line Description': df_copy['Memo'],
                    'Name': df_copy['Relationship Name'] if 'Relationship Name' in df_copy.columns else '',
                    'Debit Amount': df_copy['Debit'],
                    'Credit Amount': df_copy['Credit']
                })
                
                processed_sheets.append(processed_df)
                
            except Exception as e:
                self.print_and_log(f'Error processing sheet {sheet_name}: {str(e)}')
                continue
        
        if not processed_sheets:
            raise Exception('No sheets were successfully processed')
        
        journal_entries = pd.concat(processed_sheets, ignore_index=True)
        self.print_and_log(f'✅ Journal entries created: {len(journal_entries)} entries')
        return journal_entries

    def create_output_file(self, trial_balance, journal_entries):
        """Create Excel output file with professional formatting"""
        self.update_status('Creating Excel output...', 80)
        
        start_str = self.start_date.strftime('%Y%m%d')
        end_str = self.end_date.strftime('%Y%m%d')
        self.output_filename = f'Processed_Strongbox_{start_str}_{end_str}.xlsx'
        
        # Create Excel workbook with proper styling
        workbook, styles = self._create_excel_workbook()
        
        # Create main data sheets first (in desired order)
        tb_sheet = self._create_trial_balance_sheet(workbook, trial_balance, styles)
        je_sheet = self._create_journal_entries_sheet(workbook, journal_entries, styles)
        
        # Create other sheets after main data sheets
        self._create_other_sheets(workbook, styles)
        
        # Save workbook
        workbook.save(self.output_filename)
        workbook.close()
        
        self.print_and_log(f'✅ Output file created: {self.output_filename}')
        return self.output_filename
    
    def _create_excel_workbook(self):
        """Create Excel workbook with basic styling setup"""
        workbook = openpyxl.Workbook()
        
        # Define styles
        from openpyxl.styles import Font, PatternFill, Alignment
        
        header_font = Font(name='Arial', size=12, bold=True, color='FFFFFF')
        blue_fill = PatternFill(start_color='0070C0', end_color='0070C0', fill_type='solid')
        gray_fill = PatternFill(start_color='999999', end_color='999999', fill_type='solid')
        dark_blue_fill = PatternFill(start_color='002060', end_color='002060', fill_type='solid')
        center_alignment = Alignment(horizontal='center', vertical='center')
        
        styles = {
            'header_font': header_font,
            'blue_fill': blue_fill,
            'gray_fill': gray_fill,
            'dark_blue_fill': dark_blue_fill,
            'center_alignment': center_alignment
        }
        
        # Remove default sheet
        if 'Sheet' in workbook.sheetnames:
            workbook.remove(workbook['Sheet'])
        
        return workbook, styles
    
    def _create_trial_balance_sheet(self, workbook, trial_balance, styles):
        """Create and format the Comparative Trial Balances sheet"""
        # Create trial balance sheet
        tb_sheet = workbook.create_sheet('Comparative Trial Balances')
        
        # Set column widths for trial balance (converting px to Excel units)
        tb_sheet.column_dimensions['A'].width = 14.3  # 100px
        tb_sheet.column_dimensions['B'].width = 34.3  # 240px
        tb_sheet.column_dimensions['C'].width = 15.7  # 110px
        tb_sheet.column_dimensions['D'].width = 15.7  # 110px
        tb_sheet.column_dimensions['E'].width = 15.7  # 110px
        tb_sheet.column_dimensions['F'].width = 34.3  # 240px
        tb_sheet.column_dimensions['G'].width = 34.3  # 240px
        
        # Write headers
        tb_sheet.append(['Required'] * 5 + ['Optional'] * 2)
        tb_sheet.append(list(trial_balance.columns))
        
        # Style row 1 headers (Required/Optional)
        for col in range(1, 6):  # Columns A-E
            cell = tb_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['blue_fill']
            cell.alignment = styles['center_alignment']
        
        for col in range(6, 8):  # Columns F-G
            cell = tb_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['gray_fill']
            cell.alignment = styles['center_alignment']
        
        # Style row 2 headers (column names) and set row height
        tb_sheet.row_dimensions[2].height = 25  # 33px ≈ 25 points
        for col in range(1, 8):  # All columns A-G
            cell = tb_sheet.cell(row=2, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['dark_blue_fill']
            cell.alignment = styles['center_alignment']
        
        # Write trial balance data with error handling
        for _, row in trial_balance.iterrows():
            row_values = []
            for col in trial_balance.columns:
                value = row[col]
                try:
                    # Double-check the value is clean
                    if isinstance(value, str) and len(value) > 1000:
                        value = value[:1000]
                    row_values.append(value)
                except:
                    row_values.append("ERROR")
            tb_sheet.append(row_values)
        
        return tb_sheet
    
    def _create_journal_entries_sheet(self, workbook, journal_entries, styles):
        """Create and format the Journal Entries & Lines sheet"""
        # Create journal entries sheet
        je_sheet = workbook.create_sheet('Journal Entries & Lines')
        
        # Set column widths for journal entries (converting px to Excel units)
        je_sheet.column_dimensions['A'].width = 14.3  # 100px
        je_sheet.column_dimensions['B'].width = 48.6  # 340px
        je_sheet.column_dimensions['C'].width = 15.7  # 110px
        je_sheet.column_dimensions['D'].width = 20.0  # 140px
        je_sheet.column_dimensions['E'].width = 35.7  # 250px
        je_sheet.column_dimensions['F'].width = 15.7  # 110px
        je_sheet.column_dimensions['G'].width = 15.7  # 110px
        je_sheet.column_dimensions['H'].width = 15.7  # 110px
        je_sheet.column_dimensions['I'].width = 15.7  # 110px
        
        # Write headers
        je_sheet.append(['Required', 'Optional', 'Optional', 'Required', 'Required', 'Optional', 'Optional', 'Required', 'Required'])
        je_sheet.append(list(journal_entries.columns))
        
        # Style row 1 headers (Required/Optional)
        required_cols = [1, 4, 5, 8, 9]  # Columns A, D, E, H, I
        optional_cols = [2, 3, 6, 7]  # Columns B, C, F, G
        
        for col in required_cols:
            cell = je_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['blue_fill']
            cell.alignment = styles['center_alignment']
        
        for col in optional_cols:
            cell = je_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['gray_fill']
            cell.alignment = styles['center_alignment']
        
        # Style row 2 headers (column names)
        for col in range(1, 10):  # All columns
            cell = je_sheet.cell(row=2, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['dark_blue_fill']
            cell.alignment = styles['center_alignment']
        
        # Write journal entries data with error handling
        for _, row in journal_entries.iterrows():
            row_values = []
            for col in journal_entries.columns:
                value = row[col]
                try:
                    # Double-check the value is clean
                    if isinstance(value, str) and len(value) > 1000:
                        value = value[:1000]
                    row_values.append(value)
                except:
                    row_values.append("ERROR")
            je_sheet.append(row_values)
        
        # Apply date formatting to column D (Posted Date) in Journal Entries & Lines
        for row_num in range(3, je_sheet.max_row + 1):  # Start from row 3 (after headers)
            cell = je_sheet.cell(row=row_num, column=4)
            cell.number_format = 'M/D/YYYY'
        
        return je_sheet
    
    def _create_other_sheets(self, workbook, styles):
        """Create and format all other sheets in the specified order"""
        # Create Instructions sheet
        instructions_sheet = workbook.create_sheet('Instructions')
        instructions_sheet.append(['Content for Instructions'])
        
        # Create Data Validation Tests sheet
        validation_sheet = workbook.create_sheet('Data Validation Tests')
        validation_sheet.append(['Content for Data Validation Tests'])
        
        # Create Notes sheet
        notes_sheet = workbook.create_sheet('Notes')
        notes_sheet.append(['Content for Notes'])
        
        # Create Banking Accts sheet with specific formatting
        banking_accts_sheet = workbook.create_sheet('Banking Accts')
        
        # Add headers for Banking Accts
        banking_accts_sheet.append(['Required', 'Required', 'Optional', 'Optional'])
        banking_accts_sheet.append(['Account Number', 'Account Name', 'Institution', 'Currency'])
        
        # Style row 1 headers for Banking Accts
        for col in range(1, 3):  # Columns A-B (Required)
            cell = banking_accts_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['blue_fill']
            cell.alignment = styles['center_alignment']
        
        for col in range(3, 5):  # Columns C-D (Optional)
            cell = banking_accts_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['gray_fill']
            cell.alignment = styles['center_alignment']
        
        # Style row 2 headers for Banking Accts
        for col in range(1, 5):  # All columns A-D
            cell = banking_accts_sheet.cell(row=2, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['dark_blue_fill']
            cell.alignment = styles['center_alignment']
        
        # Create Banking Txn sheet with specific formatting
        banking_txn_sheet = workbook.create_sheet('Banking Txn')
        
        # Add headers for Banking Txn
        banking_txn_sheet.append(['Required', 'Required', 'Required', 'Required'])
        banking_txn_sheet.append(['Posted Date', 'Description', 'Amount', 'Account Number'])
        
        # Style row 1 headers for Banking Txn (all Required)
        for col in range(1, 5):  # Columns A-D (all Required)
            cell = banking_txn_sheet.cell(row=1, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['blue_fill']
            cell.alignment = styles['center_alignment']
        
        # Style row 2 headers for Banking Txn
        for col in range(1, 5):  # All columns A-D
            cell = banking_txn_sheet.cell(row=2, column=col)
            cell.font = styles['header_font']
            cell.fill = styles['dark_blue_fill']
            cell.alignment = styles['center_alignment']
        
        # Create Mapping Categories sheet
        mapping_sheet = workbook.create_sheet('Mapping Categories')
        mapping_sheet.append(['Content for Mapping Categories'])

    def run(self):
        try:
            self.print_and_log('🚀 Starting Strongbox Parser...')
            
            # Determine date range from TB sheet
            self.determine_date_range()
            
            # Load transaction data with date filtering
            self.load_source_data()
            
            # Load trial balance data using openpyxl
            tb_data = self.load_trial_balance_data()
            self.source_data['TB'] = tb_data
            
            # Create outputs from real data
            trial_balance = self.create_trial_balance()
            journal_entries = self.create_journal_entries()
            
            # Create output file
            output_file = self.create_output_file(trial_balance, journal_entries)
            
            self.update_status('Processing complete!', 100)
            self.print_and_log('\n🎉 SUCCESS! Processing completed!')
            
            return output_file
            
        except Exception as e:
            self.print_and_log(f'\n❌ Error: {str(e)}')
            raise

print('✅ Complete Strongbox Parser loaded and ready!')


In [ ]:
#@title 📁 Upload Your Strongbox Excel File

print('📁 UPLOAD YOUR STRONGBOX EXCEL FILE')
print('=' * 40)
print('Requirements:')
print('  ✓ Excel file (.xlsx format)')
print('  ✓ TB sheet with trial balance data')
print('  ✓ TB-DATA sheet with balance information')
print('  ✓ TXN-FY sheets with transactions')
print('  ✓ Dates in row 4 of TB sheet')
print()
print('Click Choose Files below and select your file:')

uploaded = files.upload()

if uploaded:
    source_filename = list(uploaded.keys())[0]
    file_size = len(uploaded[source_filename])
    print(f'\n✅ SUCCESS! File uploaded:')
    print(f'   📄 Name: {source_filename}')
    print(f'   📊 Size: {file_size:,} bytes ({file_size/1024/1024:.1f} MB)')
    print(f'\n🔄 Ready to process! Run the next cell.')
else:
    print('\n❌ No file uploaded. Please try again.')
    source_filename = None

In [ ]:
#@title 🔄 Process Your File

if 'source_filename' in globals() and source_filename:
    print('🔄 PROCESSING YOUR STRONGBOX FILE')
    print('=' * 35)
    print(f'File: {source_filename}')
    print()
    
    # Check if StrongboxParserColab is defined
    try:
        StrongboxParserColab
    except NameError:
        print('❌ ERROR: StrongboxParserColab class not found!')
        print()
        print('🔧 SOLUTION:')
        print('1. Go back to Step 3: "🏗️ Load Complete Strongbox Parser"')
        print('2. Click the ▶️ button to run that cell')
        print('3. Wait for "✅ Complete Strongbox Parser loaded and ready!" message')
        print('4. Then come back and run this cell again')
        print()
        print('💡 TIP: You can also click Runtime → Run all to run all cells in order')
    else:
        # Initialize and run parser
        parser = StrongboxParserColab()
        parser.source_file = source_filename
        
        try:
            output_file = parser.run()
            
            print('\n' + '=' * 50)
            print('🎉 SUCCESS! YOUR FILE HAS BEEN PROCESSED!')
            print('=' * 50)
            print(f'✅ Output file: {output_file}')
            print('\n📋 Your file contains:')
            print('   📊 Comparative Trial Balances (Real Data from TB-DATA)')
            print('   📝 Journal Entries & Lines (Your Real Transactions)')
            print('   📋 Professional formatting and data cleaning')
            print('\n📥 Ready for download! Run the next cell.')
            
        except Exception as e:
            print('\n' + '=' * 40)
            print('❌ PROCESSING ERROR')
            print('=' * 40)
            print(f'Error: {str(e)}')
            print('\n🔍 Please check:')
            print('  • File has TB sheet with trial balance')
            print('  • File has TB-DATA sheet with balances')
            print('  • TB sheet has dates in row 4')
            print('  • File has TXN-FY sheets with transactions')
            print('  • File is not password protected')
            print('  • File format is .xlsx (not .xls)')
            output_file = None
            
else:
    print('⚠️ Please upload a file first by running the cell above.')
    output_file = None

In [ ]:
#@title 📥 Download Your Processed File

if 'output_file' in globals() and output_file:
    print('📥 DOWNLOADING YOUR PROCESSED FILE')
    print('=' * 38)
    print(f'File: {output_file}')
    print('\nThe file will download to your browser\'s Downloads folder.')
    print()
    
    # Check if file exists before downloading
    import os
    if os.path.exists(output_file):
        # Download the file
        files.download(output_file)
        
        print('✅ Download started!')
        print('\n🎉 ALL DONE!')
        print('=' * 15)
        print('Your Audit Sight formatted file is ready!')
        print('\n💡 What you got:')
        print('  • Complete trial balance with TB-DATA balances')
        print('  • All journal entries from your TXN-FY sheets')
        print('  • Proper date filtering and account classification')
        print('  • Professional Excel formatting')
        print('  • All required tabs for Audit Sight import')
        print('\n📊 Ready for Audit Sight import!')
    else:
        print('❌ Error: Output file not found!')
        print(f'Expected file: {output_file}')
    
else:
    print('⚠️ No file ready for download.')
    print('Please upload and process a file first.')